In [ ]:
# Hapus kunci instalasi lama dan masukkan yang baru dari Google
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable

# Pastikan webdriver-manager terinstal
!pip install webdriver-manager

OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,209 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 6,951 B in 2s (3,794 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for 

In [ ]:
!pip install Sastrawi
!pip install selenium newspaper3k nltk Sastrawi webdriver-manager
!pip install lxml_html_clean

In [ ]:
import time
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from newspaper import Article
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
!apt-get update
!apt-get install -y google-chrome-stable


Hit:1 https://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,949 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state informa

In [ ]:
!apt-get update
!apt-get install -y chromium-browser
!apt-get install -y chromium-chromedriver

Hit:1 https://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (2,699 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state informa

In [ ]:
def get_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless") # Wajib berjalan di latar belakang
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # Menonaktifkan pemuatan gambar agar proses scraping jauh lebih cepat
    prefs = {"profile.managed_default_content_settings.images": 2}
    chrome_options.add_experimental_option("prefs", prefs)

    # Biarkan webdriver-manager mencari Chrome asli yang baru kita instal
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    return driver

def extract_article_data(url):
    try:
        article = Article(url, language='id')
        article.download()
        article.parse()

        if len(article.text) < 200:
            return None

        return {
            "date_published": article.publish_date.strftime("%Y-%m-%d") if article.publish_date else None,
            "title": article.title,
            "content": article.text,
            "url": url
        }
    except Exception as e:
        return None

In [ ]:
# --- EKSEKUSI UTAMA ---
# TENTUKAN TANGGAL DI SINI (Format: YYYY-MM-DD)
start_date = "2021-01-01"
end_date = "2022-12-01"


date_range = pd.date_range(start=start_date, end=end_date).strftime("%Y/%m/%d").tolist()
print(f"Mulai scraping untuk {len(date_range)} hari...")

driver = get_driver()
nama_file_csv = "berita_mentah_cnbc.csv"

for date_str in date_range:
    print(f"\n--- Mengambil data untuk tanggal: {date_str} ---")
    url_indeks = f"https://www.cnbcindonesia.com/indeks?date={date_str}"

    try:
        driver.get(url_indeks)
        time.sleep(3)


        anchors = driver.find_elements("tag name", "a")
        links_hari_ini = []
        for a in anchors:
            href = a.get_attribute("href")
            if href and "/berita/" in href or "/market/" in href:
                if href.startswith("https") and href not in links_hari_ini:
                    links_hari_ini.append(href)

        print(f"Ditemukan {len(links_hari_ini)} link berita. Memulai ekstraksi...")


        data_hari_ini = []
        for i, link in enumerate(links_hari_ini[:20]):
            hasil = extract_article_data(link)
            if hasil:
                data_hari_ini.append(hasil)


        if data_hari_ini:
            df_hari_ini = pd.DataFrame(data_hari_ini)

            if not os.path.isfile(nama_file_csv):
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='w')
            else:
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='a', header=False)
            print(f" Berhasil menyimpan {len(data_hari_ini)} berita tanggal {date_str} ke CSV.")

    except Exception as e:
        print(f" Gagal memproses tanggal {date_str}: {e}")

driver.quit()
print(f"\nSelesai! Data aman di dalam file: {nama_file_csv}")

Mulai scraping untuk 700 hari...

--- Mengambil data untuk tanggal: 2021/01/01 ---
Ditemukan 2 link berita. Memulai ekstraksi...
 Berhasil menyimpan 2 berita tanggal 2021/01/01 ke CSV.

--- Mengambil data untuk tanggal: 2021/01/02 ---
Ditemukan 0 link berita. Memulai ekstraksi...

--- Mengambil data untuk tanggal: 2021/01/03 ---
Ditemukan 2 link berita. Memulai ekstraksi...
 Berhasil menyimpan 2 berita tanggal 2021/01/03 ke CSV.

--- Mengambil data untuk tanggal: 2021/01/04 ---
Ditemukan 0 link berita. Memulai ekstraksi...

--- Mengambil data untuk tanggal: 2021/01/05 ---
Ditemukan 2 link berita. Memulai ekstraksi...
 Berhasil menyimpan 2 berita tanggal 2021/01/05 ke CSV.

--- Mengambil data untuk tanggal: 2021/01/06 ---
Ditemukan 2 link berita. Memulai ekstraksi...
 Berhasil menyimpan 2 berita tanggal 2021/01/06 ke CSV.

--- Mengambil data untuk tanggal: 2021/01/07 ---
Ditemukan 2 link berita. Memulai ekstraksi...
 Berhasil menyimpan 2 berita tanggal 2021/01/07 ke CSV.

--- Mengambil 

In [ ]:
# --- EKSEKUSI UTAMA ---
# TENTUKAN TANGGAL DI SINI (Format: YYYY-MM-DD)
start_date = "2022-12-01"
end_date = "2023-12-01"


date_range = pd.date_range(start=start_date, end=end_date).strftime("%Y/%m/%d").tolist()
print(f"Mulai scraping untuk {len(date_range)} hari...")

driver = get_driver()
nama_file_csv = "berita_mentah_cnbc.csv"

for date_str in date_range:
    print(f"\n--- Mengambil data untuk tanggal: {date_str} ---")
    url_indeks = f"https://www.cnbcindonesia.com/indeks?date={date_str}"

    try:
        driver.get(url_indeks)
        time.sleep(3)

        anchors = driver.find_elements("tag name", "a")
        links_hari_ini = []
        for a in anchors:
            href = a.get_attribute("href")
            if href and "/berita/" in href or "/market/" in href:
                if href.startswith("https") and href not in links_hari_ini:
                    links_hari_ini.append(href)

        print(f"Ditemukan {len(links_hari_ini)} link berita. Memulai ekstraksi...")

        data_hari_ini = []
        for i, link in enumerate(links_hari_ini[:20]):
            hasil = extract_article_data(link)
            if hasil:
                data_hari_ini.append(hasil)

        if data_hari_ini:
            df_hari_ini = pd.DataFrame(data_hari_ini)

            if not os.path.isfile(nama_file_csv):
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='w')
            else:
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='a', header=False)
            print(f" Berhasil menyimpan {len(data_hari_ini)} berita tanggal {date_str} ke CSV.")

    except Exception as e:
        print(f" Gagal memproses tanggal {date_str}: {e}")

driver.quit()
print(f"\nSelesai! Data aman di dalam file: {nama_file_csv}")

Mulai scraping untuk 366 hari...

--- Mengambil data untuk tanggal: 2022/12/01 ---
Ditemukan 1 link berita. Memulai ekstraksi...
 Berhasil menyimpan 1 berita tanggal 2022/12/01 ke CSV.

--- Mengambil data untuk tanggal: 2022/12/02 ---
Ditemukan 1 link berita. Memulai ekstraksi...
 Berhasil menyimpan 1 berita tanggal 2022/12/02 ke CSV.

--- Mengambil data untuk tanggal: 2022/12/03 ---
Ditemukan 0 link berita. Memulai ekstraksi...

--- Mengambil data untuk tanggal: 2022/12/04 ---
Ditemukan 3 link berita. Memulai ekstraksi...
 Berhasil menyimpan 3 berita tanggal 2022/12/04 ke CSV.

--- Mengambil data untuk tanggal: 2022/12/05 ---
Ditemukan 3 link berita. Memulai ekstraksi...
 Berhasil menyimpan 3 berita tanggal 2022/12/05 ke CSV.

--- Mengambil data untuk tanggal: 2022/12/06 ---
Ditemukan 1 link berita. Memulai ekstraksi...
 Berhasil menyimpan 1 berita tanggal 2022/12/06 ke CSV.

--- Mengambil data untuk tanggal: 2022/12/07 ---
Ditemukan 1 link berita. Memulai ekstraksi...
 Berhasil menyi

In [ ]:
# --- EKSEKUSI UTAMA ---
# TENTUKAN TANGGAL DI SINI (Format: YYYY-MM-DD)
start_date = "2023-12-01"
end_date = "2024-12-01"


date_range = pd.date_range(start=start_date, end=end_date).strftime("%Y/%m/%d").tolist()
print(f"Mulai scraping untuk {len(date_range)} hari...")

driver = get_driver()
nama_file_csv = "berita_mentah_cnbc.csv"

for date_str in date_range:
    print(f"\n--- Mengambil data untuk tanggal: {date_str} ---")
    url_indeks = f"https://www.cnbcindonesia.com/indeks?date={date_str}"

    try:
        driver.get(url_indeks)
        time.sleep(3)

        anchors = driver.find_elements("tag name", "a")
        links_hari_ini = []
        for a in anchors:
            href = a.get_attribute("href")
            if href and "/berita/" in href or "/market/" in href:
                if href.startswith("https") and href not in links_hari_ini:
                    links_hari_ini.append(href)

        print(f"Ditemukan {len(links_hari_ini)} link berita. Memulai ekstraksi...")


        data_hari_ini = []
        for i, link in enumerate(links_hari_ini[:20]):
            hasil = extract_article_data(link)
            if hasil:
                data_hari_ini.append(hasil)

        if data_hari_ini:
            df_hari_ini = pd.DataFrame(data_hari_ini)

            if not os.path.isfile(nama_file_csv):
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='w')
            else:
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='a', header=False)
            print(f"Berhasil menyimpan {len(data_hari_ini)} berita tanggal {date_str} ke CSV.")

    except Exception as e:
        print(f" Gagal memproses tanggal {date_str}: {e}")

driver.quit()
print(f"\nSelesai! Data aman di dalam file: {nama_file_csv}")

Mulai scraping untuk 367 hari...

--- Mengambil data untuk tanggal: 2023/12/01 ---
Ditemukan 2 link berita. Memulai ekstraksi...
Berhasil menyimpan 2 berita tanggal 2023/12/01 ke CSV.

--- Mengambil data untuk tanggal: 2023/12/02 ---
Ditemukan 3 link berita. Memulai ekstraksi...
Berhasil menyimpan 3 berita tanggal 2023/12/02 ke CSV.

--- Mengambil data untuk tanggal: 2023/12/03 ---
Ditemukan 0 link berita. Memulai ekstraksi...

--- Mengambil data untuk tanggal: 2023/12/04 ---
Ditemukan 1 link berita. Memulai ekstraksi...
Berhasil menyimpan 1 berita tanggal 2023/12/04 ke CSV.

--- Mengambil data untuk tanggal: 2023/12/05 ---
Ditemukan 1 link berita. Memulai ekstraksi...
Berhasil menyimpan 1 berita tanggal 2023/12/05 ke CSV.

--- Mengambil data untuk tanggal: 2023/12/06 ---
Ditemukan 0 link berita. Memulai ekstraksi...

--- Mengambil data untuk tanggal: 2023/12/07 ---
Ditemukan 2 link berita. Memulai ekstraksi...
Berhasil menyimpan 2 berita tanggal 2023/12/07 ke CSV.

--- Mengambil data 

In [ ]:
# --- EKSEKUSI UTAMA ---
# TENTUKAN TANGGAL DI SINI (Format: YYYY-MM-DD)
start_date = "2024-12-01"
end_date = "2025-12-01"


date_range = pd.date_range(start=start_date, end=end_date).strftime("%Y/%m/%d").tolist()
print(f"Mulai scraping untuk {len(date_range)} hari...")

driver = get_driver()
nama_file_csv = "berita_mentah_cnbc.csv"

for date_str in date_range:
    print(f"\n--- Mengambil data untuk tanggal: {date_str} ---")
    url_indeks = f"https://www.cnbcindonesia.com/indeks?date={date_str}"

    try:
        driver.get(url_indeks)
        time.sleep(3)


        anchors = driver.find_elements("tag name", "a")
        links_hari_ini = []
        for a in anchors:
            href = a.get_attribute("href")
            if href and "/berita/" in href or "/market/" in href:
                if href.startswith("https") and href not in links_hari_ini:
                    links_hari_ini.append(href)

        print(f"Ditemukan {len(links_hari_ini)} link berita. Memulai ekstraksi...")


        data_hari_ini = []
        for i, link in enumerate(links_hari_ini[:20]):
            hasil = extract_article_data(link)
            if hasil:
                data_hari_ini.append(hasil)


        if data_hari_ini:
            df_hari_ini = pd.DataFrame(data_hari_ini)

            if not os.path.isfile(nama_file_csv):
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='w')
            else:
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='a', header=False)
            print(f"Berhasil menyimpan {len(data_hari_ini)} berita tanggal {date_str} ke CSV.")

    except Exception as e:
        print(f"Gagal memproses tanggal {date_str}: {e}")

driver.quit()
print(f"\nSelesai! Data aman di dalam file: {nama_file_csv}")

Mulai scraping untuk 366 hari...

--- Mengambil data untuk tanggal: 2024/12/01 ---
Ditemukan 1 link berita. Memulai ekstraksi...
Berhasil menyimpan 1 berita tanggal 2024/12/01 ke CSV.

--- Mengambil data untuk tanggal: 2024/12/02 ---
Ditemukan 1 link berita. Memulai ekstraksi...
Berhasil menyimpan 1 berita tanggal 2024/12/02 ke CSV.

--- Mengambil data untuk tanggal: 2024/12/03 ---
Ditemukan 2 link berita. Memulai ekstraksi...
Berhasil menyimpan 2 berita tanggal 2024/12/03 ke CSV.

--- Mengambil data untuk tanggal: 2024/12/04 ---
Ditemukan 1 link berita. Memulai ekstraksi...
Berhasil menyimpan 1 berita tanggal 2024/12/04 ke CSV.

--- Mengambil data untuk tanggal: 2024/12/05 ---
Ditemukan 2 link berita. Memulai ekstraksi...
Berhasil menyimpan 2 berita tanggal 2024/12/05 ke CSV.

--- Mengambil data untuk tanggal: 2024/12/06 ---
Ditemukan 0 link berita. Memulai ekstraksi...

--- Mengambil data untuk tanggal: 2024/12/07 ---
Ditemukan 3 link berita. Memulai ekstraksi...
Berhasil menyimpan 3

In [ ]:

# TENTUKAN TANGGAL DI SINI (Format: YYYY-MM-DD)
start_date = "2025-12-01"
end_date = "2026-3-01" # Coba 1 minggu dulu sebagai tes

# Membuat daftar tanggal
date_range = pd.date_range(start=start_date, end=end_date).strftime("%Y/%m/%d").tolist()
print(f"Mulai scraping untuk {len(date_range)} hari...")

driver = get_driver()
nama_file_csv = "berita_mentah_cnbc.csv"

for date_str in date_range:
    print(f"\n--- Mengambil data untuk tanggal: {date_str} ---")
    url_indeks = f"https://www.cnbcindonesia.com/indeks?date={date_str}"

    try:
        driver.get(url_indeks)
        time.sleep(3) # Tunggu loading halaman

        # Kumpulkan link berita
        anchors = driver.find_elements("tag name", "a")
        links_hari_ini = []
        for a in anchors:
            href = a.get_attribute("href")
            if href and "/berita/" in href or "/market/" in href:
                if href.startswith("https") and href not in links_hari_ini:
                    links_hari_ini.append(href)

        print(f"Ditemukan {len(links_hari_ini)} link berita. Memulai ekstraksi...")

        # Proses setiap artikel di hari tersebut
        data_hari_ini = []
        for i, link in enumerate(links_hari_ini[:20]): # Opsional: Batasi 20 berita per hari agar cepat
            hasil = extract_article_data(link)
            if hasil:
                data_hari_ini.append(hasil)

        # AUTO-SAVE: Langsung simpan ke CSV setiap selesai 1 hari!
        if data_hari_ini:
            df_hari_ini = pd.DataFrame(data_hari_ini)
            # Jika file belum ada, buat baru dengan header. Jika sudah ada, tambahkan datanya di bawah (append)
            if not os.path.isfile(nama_file_csv):
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='w')
            else:
                df_hari_ini.to_csv(nama_file_csv, index=False, mode='a', header=False)
            print(f"Berhasil menyimpan {len(data_hari_ini)} berita tanggal {date_str} ke CSV.")

    except Exception as e:
        print(f"Gagal memproses tanggal {date_str}: {e}")

driver.quit()
print(f"\nSelesai! Data aman di dalam file: {nama_file_csv}")

Mulai scraping untuk 91 hari...

--- Mengambil data untuk tanggal: 2025/12/01 ---
Ditemukan 0 link berita. Memulai ekstraksi...

--- Mengambil data untuk tanggal: 2025/12/02 ---
Ditemukan 0 link berita. Memulai ekstraksi...

--- Mengambil data untuk tanggal: 2025/12/03 ---
Ditemukan 2 link berita. Memulai ekstraksi...
Berhasil menyimpan 2 berita tanggal 2025/12/03 ke CSV.

--- Mengambil data untuk tanggal: 2025/12/04 ---
Ditemukan 1 link berita. Memulai ekstraksi...
Berhasil menyimpan 1 berita tanggal 2025/12/04 ke CSV.

--- Mengambil data untuk tanggal: 2025/12/05 ---
Ditemukan 0 link berita. Memulai ekstraksi...

--- Mengambil data untuk tanggal: 2025/12/06 ---
Ditemukan 1 link berita. Memulai ekstraksi...
Berhasil menyimpan 1 berita tanggal 2025/12/06 ke CSV.

--- Mengambil data untuk tanggal: 2025/12/07 ---
Ditemukan 2 link berita. Memulai ekstraksi...
Berhasil menyimpan 2 berita tanggal 2025/12/07 ke CSV.

--- Mengambil data untuk tanggal: 2025/12/08 ---
Ditemukan 0 link berita. M

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')

print("Memuat data dari CSV...")

df_berita = pd.read_csv("berita_mentah_cnbc.csv")

df_berita = df_berita.dropna(subset=['date_published'])
print(f"Total berita yang akan diproses: {len(df_berita)} artikel")


factory = StemmerFactory()
stemmer = factory.create_stemmer()

def bersihkan_teks(teks):
    if not isinstance(teks, str):
        return ""

    tokens = word_tokenize(teks)

    teks_bersih = " ".join(tokens)
    return stemmer.stem(teks_bersih)

print("Memulai proses Stemming Sastrawi (Ini mungkin memakan waktu)...")

df_sample = df_berita.head(5).copy()

df_sample['content_clean'] = df_sample['content'].apply(bersihkan_teks)

print("\nHasil Preprocessing:")
print(df_sample[['title', 'content_clean']])

df_berita['content_clean'] = df_berita['content'].apply(bersihkan_teks)
df_berita.to_csv("berita_bersih_sastrawi.csv", index=False)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Memuat data dari CSV...
Total berita yang akan diproses: 2286 artikel
Memulai proses Stemming Sastrawi (Ini mungkin memakan waktu)...

Hasil Preprocessing:
                                               title  \
0           Awas Bubble! Saham Tesla Sudah Kemahalan   
1  Emas Boleh Cuan 25%, Tapi Ada yang Lebih 'Gila...   
2  Sederet Faktor yang Membuat BRI Optimis Songso...   
3  BTN Guyur Subsidi Bunga KPR Rp 2,49 T Hingga A...   
4  Senat Diprediksi Tetap Sehat, Wall Street Dibu...   

                                       content_clean  
0  jakarta cnbc indonesia - saham emiten mobil li...  
1  jakarta cnbc indonesia - emas memang jadi sala...  
2  jakarta cnbc indonesia - pt bank rakyat indone...  
3  jakarta cnbc indonesia - pt bank tabung negara...  
4  jakarta cnbc indonesia - bursa saham amerika s...  


In [ ]:


# Muat data berita mentah
df_mentah = pd.read_csv('berita_mentah_cnbc.csv')

print("\n--- Data Berita Mentah (Raw Data) ---")
display(df_mentah.head())
print("\nDeskripsi Statistik Data Mentah:")
display(df_mentah.describe(include='all'))


--- Data Berita Mentah (Raw Data) ---


,date_published,title,content,url
0,2021-01-01,Awas Bubble! Saham Tesla Sudah Kemahalan,"Jakarta, CNBC Indonesia - Saham emiten mobil l...",https://www.cnbcindonesia.com/market/202101011...
1,2021-01-01,"Emas Boleh Cuan 25%, Tapi Ada yang Lebih 'Gila...","Jakarta, CNBC Indonesia - Emas memang menjadi ...",https://www.cnbcindonesia.com/market/202101011...
2,2021-01-03,Sederet Faktor yang Membuat BRI Optimis Songso...,"Jakarta, CNBC Indonesia - PT. Bank Rakyat Indo...",https://www.cnbcindonesia.com/market/202101032...
3,2021-01-03,"BTN Guyur Subsidi Bunga KPR Rp 2,49 T Hingga A...","Jakarta, CNBC Indonesia - PT Bank Tabungan Neg...",https://www.cnbcindonesia.com/market/202101031...
4,2021-01-05,"Senat Diprediksi Tetap Sehat, Wall Street Dibu...","Jakarta, CNBC Indonesia - Bursa saham Amerika ...",https://www.cnbcindonesia.com/market/202101052...



Deskripsi Statistik Data Mentah:


,date_published,title,content,url
count,2286,2286,2286,2286
unique,1359,2273,2281,2282
top,2023-07-25,"Tertarik Punya Alfamart Sendiri, Segini Biaya ...","Jakarta, CNBC Indonesia - Bursa saham Amerika ...",https://www.cnbcindonesia.com/market/202212012...
freq,8,3,2,2


In [ ]:
# Muat data berita yang sudah dibersihkan
df_bersih = pd.read_csv('berita_bersih_sastrawi.csv')

print("\n--- Data Berita yang Sudah Dibersihkan (Cleaned Data) ---")
display(df_bersih.head())
print("\nDeskripsi Statistik Data Bersih:")
display(df_bersih.describe(include='all'))


--- Data Berita yang Sudah Dibersihkan (Cleaned Data) ---


,date_published,title,content,url,content_clean
0,2021-01-01,Awas Bubble! Saham Tesla Sudah Kemahalan,"Jakarta, CNBC Indonesia - Saham emiten mobil l...",https://www.cnbcindonesia.com/market/202101011...,jakarta cnbc indonesia - saham emiten mobil li...
1,2021-01-01,"Emas Boleh Cuan 25%, Tapi Ada yang Lebih 'Gila...","Jakarta, CNBC Indonesia - Emas memang menjadi ...",https://www.cnbcindonesia.com/market/202101011...,jakarta cnbc indonesia - emas memang jadi sala...
2,2021-01-03,Sederet Faktor yang Membuat BRI Optimis Songso...,"Jakarta, CNBC Indonesia - PT. Bank Rakyat Indo...",https://www.cnbcindonesia.com/market/202101032...,jakarta cnbc indonesia - pt bank rakyat indone...
3,2021-01-03,"BTN Guyur Subsidi Bunga KPR Rp 2,49 T Hingga A...","Jakarta, CNBC Indonesia - PT Bank Tabungan Neg...",https://www.cnbcindonesia.com/market/202101031...,jakarta cnbc indonesia - pt bank tabung negara...
4,2021-01-05,"Senat Diprediksi Tetap Sehat, Wall Street Dibu...","Jakarta, CNBC Indonesia - Bursa saham Amerika ...",https://www.cnbcindonesia.com/market/202101052...,jakarta cnbc indonesia - bursa saham amerika s...



Deskripsi Statistik Data Bersih:


,date_published,title,content,url,content_clean
count,2286,2286,2286,2286,2286
unique,1359,2273,2281,2282,2281
top,2023-07-25,"Tertarik Punya Alfamart Sendiri, Segini Biaya ...","Jakarta, CNBC Indonesia - Bursa saham Amerika ...",https://www.cnbcindonesia.com/market/202212012...,jakarta cnbc indonesia - bursa saham amerika s...
freq,8,3,2,2,2


In [ ]:
print("\n--- Perbandingan Konten (Mentah vs Bersih) ---")
# Ambil beberapa sampel untuk perbandingan
comparison_df = df_mentah[['title', 'content']].head(5).copy()
comparison_df['content_clean'] = df_bersih['content_clean'].head(5)
display(comparison_df[['title', 'content', 'content_clean']])


--- Perbandingan Konten (Mentah vs Bersih) ---


,title,content,content_clean
0,Awas Bubble! Saham Tesla Sudah Kemahalan,"Jakarta, CNBC Indonesia - Saham emiten mobil l...",jakarta cnbc indonesia - saham emiten mobil li...
1,"Emas Boleh Cuan 25%, Tapi Ada yang Lebih 'Gila...","Jakarta, CNBC Indonesia - Emas memang menjadi ...",jakarta cnbc indonesia - emas memang jadi sala...
2,Sederet Faktor yang Membuat BRI Optimis Songso...,"Jakarta, CNBC Indonesia - PT. Bank Rakyat Indo...",jakarta cnbc indonesia - pt bank rakyat indone...
3,"BTN Guyur Subsidi Bunga KPR Rp 2,49 T Hingga A...","Jakarta, CNBC Indonesia - PT Bank Tabungan Neg...",jakarta cnbc indonesia - pt bank tabung negara...
4,"Senat Diprediksi Tetap Sehat, Wall Street Dibu...","Jakarta, CNBC Indonesia - Bursa saham Amerika ...",jakarta cnbc indonesia - bursa saham amerika s...
